In [1]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd

np.random.seed(42)

project_root = Path("..")
input_file = project_root / "data" / "processed" / "modelling_dataset.parquet"
out_tables = project_root / "outputs" / "tables" / "step15B"
out_results = project_root / "outputs" / "model_results"
out_tables.mkdir(parents=True, exist_ok=True)
out_results.mkdir(parents=True, exist_ok=True)

ALPHA = 0.1

model_data = pd.read_parquet(input_file)
model_data["date"] = pd.to_datetime(model_data["date"])
model_data = model_data.sort_values(["id", "date"]).reset_index(drop=True)

demand_type_order = ["smooth", "erratic", "intermittent", "lumpy"]
model_data["demand_type"] = pd.Categorical(
    model_data["demand_type"], categories=demand_type_order, ordered=True
)

print("Rows:", len(model_data))
print("Series:", model_data["id"].nunique())
print(model_data[["id","demand_type"]].drop_duplicates()["demand_type"].value_counts(sort=False))

Rows: 1552800
Series: 800
demand_type
smooth          200
erratic         200
intermittent    200
lumpy           200
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import TimeSeriesSplit

unique_dates = np.array(sorted(model_data["date"].unique()))
time_split = TimeSeriesSplit(n_splits=5, test_size=28, gap=0)
date_index_input = np.arange(len(unique_dates)).reshape(-1, 1)

fold_date_indices = []
for fold, (tr, te) in enumerate(time_split.split(date_index_input), start=1):
    fold_date_indices.append({
        "fold": fold,
        "train_dates": unique_dates[tr],
        "test_dates": unique_dates[te],
    })

print("Folds created:", len(fold_date_indices))
for f in fold_date_indices:
    print("Fold", f["fold"], ":", pd.Timestamp(f["test_dates"][0]).date(),
          "to", pd.Timestamp(f["test_dates"][-1]).date())

Folds created: 5
Fold 1 : 2016-01-04 to 2016-01-31
Fold 2 : 2016-02-01 to 2016-02-28
Fold 3 : 2016-02-29 to 2016-03-27
Fold 4 : 2016-03-28 to 2016-04-24
Fold 5 : 2016-04-25 to 2016-05-22


In [3]:
hierarchy_levels = [
    ("L1_Total", []), ("L2_State", ["state_id"]), ("L3_Store", ["store_id"]),
    ("L4_Category", ["cat_id"]), ("L5_Department", ["dept_id"]),
    ("L6_State_Category", ["state_id","cat_id"]), ("L7_State_Department", ["state_id","dept_id"]),
    ("L8_Store_Category", ["store_id","cat_id"]), ("L9_Store_Department", ["store_id","dept_id"]),
    ("L10_Item", ["item_id"]), ("L11_Item_State", ["item_id","state_id"]),
    ("L12_Item_Store", ["item_id","store_id"]),
]

def create_series_key(data, cols):
    return data[cols].astype("string").agg("|".join, axis=1)

def aggregate_daily_values(data, cols, value_columns):
    if len(cols) == 0:
        agg = data.groupby("date", as_index=False, sort=True)[value_columns].sum()
        agg["series_key"] = "Total"
    else:
        agg = data.groupby(cols + ["date"], observed=True, as_index=False, sort=True)[value_columns].sum()
        agg["series_key"] = create_series_key(agg, cols)
    return agg[["series_key","date"] + value_columns]

def aggregate_dollar_weights(data, cols):
    if len(cols) == 0:
        return pd.DataFrame({"series_key":["Total"], "raw_weight":[data["dollar_sales"].sum()]})
    agg = data.groupby(cols, observed=True, as_index=False, sort=True)["dollar_sales"].sum()
    agg = agg.rename(columns={"dollar_sales":"raw_weight"})
    agg["series_key"] = create_series_key(agg, cols)
    return agg[["series_key","raw_weight"]]

def calculate_m5_scale(values):
    values = np.asarray(values, dtype=np.float64)
    nz = np.flatnonzero(values != 0)
    if len(nz) == 0:
        return np.nan
    trimmed = values[nz[0]:]
    if len(trimmed) < 2:
        return np.nan
    scale = (np.diff(trimmed) ** 2).mean()
    if not np.isfinite(scale) or scale <= 0:
        return np.nan
    return float(scale)

def evaluate_type_wrmsse(train_data, test_data, demand_type, fold):
    train_end = train_data["date"].max()
    weight_start = train_end - pd.Timedelta(days=27)
    ww = train_data.loc[train_data["date"].between(weight_start, train_end)].copy()
    bad = ((ww["sales"] > 0) & (ww["sell_price"].isna())).sum()
    if bad > 0:
        raise ValueError(f"{bad} positive-sales rows have missing prices in weight window.")
    ww["dollar_sales"] = ww["sales"].astype("float64") * ww["sell_price"].fillna(0).astype("float64")

    diags = []
    for level_name, cols in hierarchy_levels:
        tr_lvl = aggregate_daily_values(train_data, cols, ["sales"])
        te_lvl = aggregate_daily_values(test_data, cols, ["sales","naive_forecast"])
        wts = aggregate_dollar_weights(ww, cols)
        scale_tbl = (tr_lvl.sort_values(["series_key","date"])
                     .groupby("series_key", sort=False)["sales"]
                     .apply(calculate_m5_scale).rename("scale").reset_index())
        te_lvl["squared_error"] = (te_lvl["sales"] - te_lvl["naive_forecast"]) ** 2
        te_lvl["absolute_error"] = (te_lvl["sales"] - te_lvl["naive_forecast"]).abs()
        err = (te_lvl.groupby("series_key", as_index=False, sort=False)
               .agg(test_mse=("squared_error","mean"),
                    test_mae=("absolute_error","mean"),
                    test_days=("date","nunique")))
        lr = (err.merge(scale_tbl, on="series_key", how="left", validate="one_to_one")
                  .merge(wts, on="series_key", how="left", validate="one_to_one"))
        invalid = lr["scale"].isna() & lr["raw_weight"].gt(0)
        if invalid.any():
            raise ValueError(f"{demand_type}, Fold {fold}, {level_name}: {int(invalid.sum())} positive-weight series have invalid scale.")
        lr["rmsse"] = np.where(lr["scale"].gt(0), np.sqrt(lr["test_mse"] / lr["scale"]), 0.0)
        diags.append(lr)
        del tr_lvl, te_lvl, scale_tbl, err, lr
        gc.collect()

    d = pd.concat(diags, ignore_index=True)
    total_w = d["raw_weight"].sum()
    if total_w <= 0:
        raise ValueError(f"{demand_type}, Fold {fold}: total weight is zero.")
    d["weight"] = d["raw_weight"] / total_w
    d["weighted_rmsse"] = d["weight"] * d["rmsse"]
    return float(d["weighted_rmsse"].sum())

print("Scorer ready.")

Scorer ready.


In [4]:
def croston_single(sales_values, alpha=ALPHA, method="croston"):
    sales = np.asarray(sales_values, dtype=np.float64)
    nz = np.flatnonzero(sales > 0)
    if len(nz) == 0:
        return 0.0
    z = float(sales[nz[0]])
    p = float(nz[0] + 1)
    if p < 1.0:
        p = 1.0
    prev = nz[0]
    for idx in nz[1:]:
        q = float(idx - prev)
        z = alpha * float(sales[idx]) + (1.0 - alpha) * z
        p = alpha * q + (1.0 - alpha) * p
        prev = idx
    fc = z / p
    if method == "sba":
        fc = (1.0 - alpha / 2.0) * fc
    return float(fc)

def make_forecast(train_data, test_data, method):
    per_series = (train_data.sort_values(["id","date"])
                  .groupby("id", observed=True)["sales"]
                  .apply(lambda s: croston_single(s.values, method=method))
                  .rename("model_forecast").reset_index())
    fd = test_data.copy().merge(per_series, on="id", how="left", validate="many_to_one", sort=False)
    if fd["model_forecast"].isna().sum() > 0:
        raise ValueError(f"{fd['model_forecast'].isna().sum()} test rows missing {method} forecast.")
    fd["naive_forecast"] = fd["model_forecast"].astype("float64")
    return fd

print("Croston/SBA ready.")

Croston/SBA ready.


In [5]:
records = []
for method in ["croston", "sba"]:
    print("\n" + "="*40 + "\n" + method.upper() + "\n" + "="*40)
    for fi in fold_date_indices:
        fold = fi["fold"]
        train_end = pd.Timestamp(fi["train_dates"][-1])
        test_start = pd.Timestamp(fi["test_dates"][0])
        test_end = pd.Timestamp(fi["test_dates"][-1])
        train_fold = model_data.loc[model_data["date"] <= train_end].copy()
        raw_test = model_data.loc[model_data["date"].between(test_start, test_end)].copy()
        test_fold = make_forecast(train_fold, raw_test, method)
        print(f"\nFold {fold}: {test_start.date()} to {test_end.date()}")
        for dt in demand_type_order:
            tr = train_fold.loc[train_fold["demand_type"] == dt].copy()
            te = test_fold.loc[test_fold["demand_type"] == dt].copy()
            if te["id"].nunique() != 200:
                raise ValueError(f"{dt} Fold {fold}: expected 200 series, found {te['id'].nunique()}.")
            mae = (te["sales"] - te["naive_forecast"]).abs().mean()
            wr = evaluate_type_wrmsse(tr, te, dt, fold)
            records.append({"method":method, "fold":fold, "demand_type":dt,
                            "wrmsse":wr, "mae":float(mae)})
            print(f"  {dt}: WRMSSE={wr:.4f}, MAE={mae:.4f}")
            del tr, te; gc.collect()
        del train_fold, raw_test, test_fold; gc.collect()

print("\nDONE")


CROSTON

Fold 1: 2016-01-04 to 2016-01-31
  smooth: WRMSSE=0.8970, MAE=3.0843
  erratic: WRMSSE=0.9781, MAE=3.9430
  intermittent: WRMSSE=1.0045, MAE=0.7816
  lumpy: WRMSSE=1.0482, MAE=1.5113

Fold 2: 2016-02-01 to 2016-02-28
  smooth: WRMSSE=0.8977, MAE=2.9903
  erratic: WRMSSE=0.9741, MAE=3.8351
  intermittent: WRMSSE=0.9628, MAE=0.8090
  lumpy: WRMSSE=1.1815, MAE=1.6565

Fold 3: 2016-02-29 to 2016-03-27
  smooth: WRMSSE=0.8146, MAE=2.8149
  erratic: WRMSSE=0.7907, MAE=3.6212
  intermittent: WRMSSE=1.2437, MAE=0.9765
  lumpy: WRMSSE=1.0538, MAE=1.5337

Fold 4: 2016-03-28 to 2016-04-24
  smooth: WRMSSE=0.8106, MAE=2.9167
  erratic: WRMSSE=0.8552, MAE=3.6894
  intermittent: WRMSSE=1.1090, MAE=0.8225
  lumpy: WRMSSE=1.0325, MAE=1.5814

Fold 5: 2016-04-25 to 2016-05-22
  smooth: WRMSSE=0.8581, MAE=2.9252
  erratic: WRMSSE=0.8115, MAE=3.6980
  intermittent: WRMSSE=1.2580, MAE=0.8846
  lumpy: WRMSSE=1.0210, MAE=1.5233

SBA

Fold 1: 2016-01-04 to 2016-01-31
  smooth: WRMSSE=0.9004, MAE=3.0

In [6]:
fold_results = pd.DataFrame(records)
summary = (fold_results.groupby(["method","demand_type"], observed=True, as_index=False)
           .agg(mean_wrmsse=("wrmsse","mean"),
                std_wrmsse=("wrmsse","std"),
                mean_mae=("mae","mean"))
           .sort_values(["method","demand_type"]))

print(summary.to_string(index=False))

fold_results.to_csv(out_results / "step15B_croston_sba_fold_results.csv", index=False)
summary.to_csv(out_tables / "step15B_croston_sba_summary.csv", index=False)
print("\nSaved.")

 method  demand_type  mean_wrmsse  std_wrmsse  mean_mae
croston      erratic     0.881935    0.089107  3.757337
croston intermittent     1.115587    0.134576  0.854828
croston        lumpy     1.067415    0.065099  1.561224
croston       smooth     0.855583    0.042437  2.946278
    sba      erratic     0.885256    0.116428  3.701286
    sba intermittent     1.097776    0.147690  0.841436
    sba        lumpy     1.075630    0.078990  1.537972
    sba       smooth     0.853621    0.069448  2.905852

Saved.


In [7]:
comparison = pd.DataFrame({
    "Seasonal_Naive": [0.8064, 0.9500, 1.2156, 1.2589],
    "XGBoost":        [0.6360, 0.7938, 1.2095, 1.1275],
    "Croston":        [0.8556, 0.8819, 1.1156, 1.0674],
    "SBA":            [0.8536, 0.8853, 1.0978, 1.0756],
}, index=["smooth","erratic","intermittent","lumpy"])

comparison.index.name = "demand_type"
comparison["best_model"] = comparison.idxmin(axis=1)
print(comparison.round(4).to_string())
comparison.to_csv(out_tables / "step15B_four_way_comparison.csv")
print("\nSaved four-way comparison.")

              Seasonal_Naive  XGBoost  Croston     SBA best_model
demand_type                                                      
smooth                0.8064   0.6360   0.8556  0.8536    XGBoost
erratic               0.9500   0.7938   0.8819  0.8853    XGBoost
intermittent          1.2156   1.2095   1.1156  1.0978        SBA
lumpy                 1.2589   1.1275   1.0674  1.0756    Croston

Saved four-way comparison.


In [8]:
audit_records = []
for method in ["croston", "sba"]:
    for fi in fold_date_indices:
        fold = fi["fold"]
        train_end  = pd.Timestamp(fi["train_dates"][-1])
        test_start = pd.Timestamp(fi["test_dates"][0])
        test_end   = pd.Timestamp(fi["test_dates"][-1])
        train_fold = model_data.loc[model_data["date"] <= train_end]
        raw_test   = model_data.loc[model_data["date"].between(test_start, test_end)]
        fd = make_forecast(train_fold, raw_test, method)
        train_dates_set = set(pd.to_datetime(fi["train_dates"]))
        test_dates_set  = set(pd.to_datetime(fi["test_dates"]))
        overlap = train_dates_set & test_dates_set
        missing = fd["model_forecast"].isna().sum()
        negative = (fd["model_forecast"] < 0).sum()
        max_unique = fd.groupby("id")["model_forecast"].nunique().max()
        audit_records.append({
            "method": method,
            "fold": fold,
            "train_test_overlap": len(overlap),
            "missing_forecasts": missing,
            "negative_forecasts": negative,
            "max_unique_per_series": max_unique,
        })
        del fd; gc.collect()

audit_df = pd.DataFrame(audit_records)
print(audit_df.to_string(index=False))
print()
print("ALL CHECKS PASS:", 
      (audit_df["train_test_overlap"] == 0).all() and
      (audit_df["missing_forecasts"] == 0).all() and
      (audit_df["negative_forecasts"] == 0).all() and
      (audit_df["max_unique_per_series"] == 1).all())

C:\Users\gamec\AppData\Local\Temp\ipykernel_25396\1554297009.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_unique = fd.groupby("id")["model_forecast"].nunique().max()
C:\Users\gamec\AppData\Local\Temp\ipykernel_25396\1554297009.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_unique = fd.groupby("id")["model_forecast"].nunique().max()
C:\Users\gamec\AppData\Local\Temp\ipykernel_25396\1554297009.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt t

 method  fold  train_test_overlap  missing_forecasts  negative_forecasts  max_unique_per_series
croston     1                   0                  0                   0                      1
croston     2                   0                  0                   0                      1
croston     3                   0                  0                   0                      1
croston     4                   0                  0                   0                      1
croston     5                   0                  0                   0                      1
    sba     1                   0                  0                   0                      1
    sba     2                   0                  0                   0                      1
    sba     3                   0                  0                   0                      1
    sba     4                   0                  0                   0                      1
    sba     5                   0       

C:\Users\gamec\AppData\Local\Temp\ipykernel_25396\1554297009.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_unique = fd.groupby("id")["model_forecast"].nunique().max()


In [9]:
sensitivity_records = []
for alpha_val in [0.05, 0.10, 0.15, 0.20]:
    for demand_type in ["intermittent", "lumpy"]:
        fold_wrmsse = []
        for fi in fold_date_indices:
            fold = fi["fold"]
            train_end  = pd.Timestamp(fi["train_dates"][-1])
            test_start = pd.Timestamp(fi["test_dates"][0])
            test_end   = pd.Timestamp(fi["test_dates"][-1])
            train_fold = model_data.loc[model_data["date"] <= train_end]
            raw_test   = model_data.loc[model_data["date"].between(test_start, test_end)]
            per_series = (
                train_fold.sort_values(["id","date"])
                .groupby("id", observed=True)["sales"]
                .apply(lambda s: croston_single(s.values, alpha=alpha_val, method="sba"))
                .rename("model_forecast").reset_index()
            )
            fd = raw_test.copy().merge(per_series, on="id", how="left",
                                       validate="many_to_one", sort=False)
            fd["naive_forecast"] = fd["model_forecast"].astype("float64")
            tr = train_fold.loc[train_fold["demand_type"] == demand_type].copy()
            te = fd.loc[fd["demand_type"] == demand_type].copy()
            wr = evaluate_type_wrmsse(tr, te, demand_type, fold)
            fold_wrmsse.append(wr)
            del tr, te, fd, train_fold, raw_test; gc.collect()
        sensitivity_records.append({
            "alpha": alpha_val,
            "demand_type": demand_type,
            "mean_wrmsse": round(sum(fold_wrmsse)/len(fold_wrmsse), 4)
        })

sens_df = pd.DataFrame(sensitivity_records)
print(sens_df.pivot(index="alpha", columns="demand_type", values="mean_wrmsse").to_string())

demand_type  intermittent   lumpy
alpha                            
0.05               1.1336  1.0852
0.10               1.0978  1.0756
0.15               1.0752  1.0872
0.20               1.0683  1.1026


In [10]:
rmse_records = []
for method in ["croston", "sba"]:
    for fi in fold_date_indices:
        fold = fi["fold"]
        train_end  = pd.Timestamp(fi["train_dates"][-1])
        test_start = pd.Timestamp(fi["test_dates"][0])
        test_end   = pd.Timestamp(fi["test_dates"][-1])
        train_fold = model_data.loc[model_data["date"] <= train_end]
        raw_test   = model_data.loc[model_data["date"].between(test_start, test_end)]
        fd = make_forecast(train_fold, raw_test, method)
        for dt in demand_type_order:
            te = fd.loc[fd["demand_type"] == dt]
            rmse = float(((te["sales"] - te["naive_forecast"])**2).mean()**0.5)
            mae  = float((te["sales"] - te["naive_forecast"]).abs().mean())
            rmse_records.append({
                "method": method,
                "fold": fold,
                "demand_type": dt,
                "rmse": rmse,
                "mae": mae
            })
            del te; gc.collect()
        del train_fold, raw_test, fd; gc.collect()

rmse_df = pd.DataFrame(rmse_records)
rmse_summary = (rmse_df.groupby(["method","demand_type"], observed=True)[["rmse","mae"]]
                .mean().round(4))
print(rmse_summary.to_string())
rmse_df.to_csv(out_results / "step15B_croston_sba_rmse.csv", index=False)
print("\nSaved.")

                        rmse     mae
method  demand_type                 
croston erratic       5.8746  3.7573
        intermittent  2.1485  0.8548
        lumpy         2.7270  1.5612
        smooth        4.8685  2.9463
sba     erratic       5.8535  3.7013
        intermittent  2.1179  0.8414
        lumpy         2.7175  1.5380
        smooth        4.8368  2.9059

Saved.


In [11]:
croston_lumpy_sensitivity = []
for alpha_val in [0.05, 0.10, 0.15, 0.20]:
    fold_wrmsse = []
    for fi in fold_date_indices:
        fold = fi["fold"]
        train_end  = pd.Timestamp(fi["train_dates"][-1])
        test_start = pd.Timestamp(fi["test_dates"][0])
        test_end   = pd.Timestamp(fi["test_dates"][-1])
        train_fold = model_data.loc[model_data["date"] <= train_end]
        raw_test   = model_data.loc[model_data["date"].between(test_start, test_end)]
        per_series = (
            train_fold.sort_values(["id","date"])
            .groupby("id", observed=True)["sales"]
            .apply(lambda s: croston_single(s.values, alpha=alpha_val, method="croston"))
            .rename("model_forecast").reset_index()
        )
        fd = raw_test.copy().merge(per_series, on="id", how="left",
                                   validate="many_to_one", sort=False)
        fd["naive_forecast"] = fd["model_forecast"].astype("float64")
        tr = train_fold.loc[train_fold["demand_type"] == "lumpy"].copy()
        te = fd.loc[fd["demand_type"] == "lumpy"].copy()
        wr = evaluate_type_wrmsse(tr, te, "lumpy", fold)
        fold_wrmsse.append(wr)
        del tr, te, fd, train_fold, raw_test; gc.collect()
    croston_lumpy_sensitivity.append({
        "alpha": alpha_val,
        "method": "croston",
        "demand_type": "lumpy",
        "mean_wrmsse": round(sum(fold_wrmsse)/len(fold_wrmsse), 4)
    })
    print(f"Croston alpha={alpha_val} lumpy WRMSSE: {round(sum(fold_wrmsse)/5, 4)}")

print("\nCroston lumpy sensitivity complete.")

Croston alpha=0.05 lumpy WRMSSE: 1.0754
Croston alpha=0.1 lumpy WRMSSE: 1.0674
Croston alpha=0.15 lumpy WRMSSE: 1.0759
Croston alpha=0.2 lumpy WRMSSE: 1.0858

Croston lumpy sensitivity complete.


In [12]:
def evaluate_with_levels(train_data, test_data, demand_type, fold):
    """Modified scorer that returns both scalar WRMSSE and per-level breakdown."""
    train_end = train_data["date"].max()
    weight_start = train_end - pd.Timedelta(days=27)
    ww = train_data.loc[train_data["date"].between(weight_start, train_end)].copy()
    ww["dollar_sales"] = (ww["sales"].astype("float64") *
                          ww["sell_price"].fillna(0).astype("float64"))

    level_rows = []
    for level_name, cols in hierarchy_levels:
        tr_lvl = aggregate_daily_values(train_data, cols, ["sales"])
        te_lvl = aggregate_daily_values(test_data, cols, ["sales","naive_forecast"])
        wts    = aggregate_dollar_weights(ww, cols)
        scale_tbl = (tr_lvl.sort_values(["series_key","date"])
                     .groupby("series_key", sort=False)["sales"]
                     .apply(calculate_m5_scale).rename("scale").reset_index())
        te_lvl["sq_err"] = (te_lvl["sales"] - te_lvl["naive_forecast"]) ** 2
        err = (te_lvl.groupby("series_key", as_index=False, sort=False)
               .agg(test_mse=("sq_err","mean")))
        lr = (err.merge(scale_tbl, on="series_key", how="left")
                  .merge(wts, on="series_key", how="left"))
        lr["rmsse"] = np.where(lr["scale"].gt(0),
                               np.sqrt(lr["test_mse"]/lr["scale"]), 0.0)
        total_w = lr["raw_weight"].sum()
        if total_w > 0:
            lr["weight"] = lr["raw_weight"] / total_w
        else:
            lr["weight"] = 0.0
        lr["weighted_rmsse"] = lr["weight"] * lr["rmsse"]
        level_rows.append({
            "level": level_name,
            "wrmsse": lr["weighted_rmsse"].sum()
        })
        del tr_lvl, te_lvl, scale_tbl, err, lr; gc.collect()

    level_df = pd.DataFrame(level_rows)
    total_wrmsse = float(level_df["wrmsse"].sum())
    return total_wrmsse, level_df

# Run hierarchy breakdown for Croston and SBA on intermittent and lumpy
hierarchy_records = []
for method in ["croston", "sba"]:
    for demand_type in ["intermittent", "lumpy"]:
        for fi in fold_date_indices:
            fold = fi["fold"]
            train_end  = pd.Timestamp(fi["train_dates"][-1])
            test_start = pd.Timestamp(fi["test_dates"][0])
            test_end   = pd.Timestamp(fi["test_dates"][-1])
            train_fold = model_data.loc[model_data["date"] <= train_end]
            raw_test   = model_data.loc[model_data["date"].between(test_start, test_end)]
            fd = make_forecast(train_fold, raw_test, method)
            tr = train_fold.loc[train_fold["demand_type"] == demand_type].copy()
            te = fd.loc[fd["demand_type"] == demand_type].copy()
            _, level_df = evaluate_with_levels(tr, te, demand_type, fold)
            for _, row in level_df.iterrows():
                hierarchy_records.append({
                    "method": method,
                    "demand_type": demand_type,
                    "fold": fold,
                    "level": row["level"],
                    "wrmsse": row["wrmsse"]
                })
            del tr, te, fd, train_fold, raw_test; gc.collect()

hier_df = pd.DataFrame(hierarchy_records)
hier_mean = (hier_df.groupby(["method","demand_type","level"])["wrmsse"]
             .mean().round(4).reset_index())

# Show intermittent and lumpy side by side vs XGBoost levels from step16
print("Mean WRMSSE by hierarchy level — Croston and SBA:")
print()
for dt in ["intermittent","lumpy"]:
    print(f"\n--- {dt.upper()} ---")
    sub = hier_mean[hier_mean["demand_type"]==dt].pivot(
        index="level", columns="method", values="wrmsse")
    sub["difference_sba_minus_croston"] = sub["sba"] - sub["croston"]
    print(sub.to_string())

hier_df.to_csv(out_results / "step15B_hierarchy_breakdown.csv", index=False)
print("\nSaved.")

Mean WRMSSE by hierarchy level — Croston and SBA:


--- INTERMITTENT ---
method               croston     sba  difference_sba_minus_croston
level                                                             
L10_Item              0.8958  0.8910                       -0.0048
L11_Item_State        0.8945  0.8898                       -0.0047
L12_Item_Store        0.8770  0.8723                       -0.0047
L1_Total              1.3758  1.3180                       -0.0578
L2_State              1.3676  1.3354                       -0.0322
L3_Store              1.1704  1.1539                       -0.0165
L4_Category           1.2576  1.2272                       -0.0304
L5_Department         1.1653  1.1487                       -0.0166
L6_State_Category     1.2124  1.1958                       -0.0166
L7_State_Department   1.1213  1.1108                       -0.0105
L8_Store_Category     1.0439  1.0343                       -0.0096
L9_Store_Department   1.0054  0.9962                    

In [13]:
import warnings
warnings.filterwarnings('ignore')

mismatch_records = []

for fi in fold_date_indices:
    fold = fi["fold"]
    train_end = pd.Timestamp(fi["train_dates"][-1])
    train_fold = model_data.loc[model_data["date"] <= train_end]

    for demand_type in demand_type_order:
        tr = train_fold.loc[train_fold["demand_type"] == demand_type].copy()

        for level_name, cols in hierarchy_levels:
            tr_lvl = aggregate_daily_values(tr, cols, ["sales"])
            scale_tbl = (
                tr_lvl.sort_values(["series_key","date"])
                .groupby("series_key", sort=False)["sales"]
                .apply(calculate_m5_scale)
                .rename("scale")
                .reset_index()
            )
            scale_tbl2 = (
                tr_lvl.sort_values(["series_key","date"])
                .groupby("series_key", sort=False)["sales"]
                .apply(calculate_m5_scale)
                .rename("scale")
                .reset_index()
            )
            max_diff = (scale_tbl["scale"] - scale_tbl2["scale"]).abs().max()
            mismatch_records.append({
                "fold": fold,
                "demand_type": demand_type,
                "level": level_name,
                "max_scale_diff": max_diff
            })
            del tr_lvl, scale_tbl, scale_tbl2; gc.collect()

mismatch_df = pd.DataFrame(mismatch_records)
overall_max = mismatch_df["max_scale_diff"].max()
print(f"Maximum denominator difference: {overall_max}")
print(f"All denominators consistent: {overall_max == 0.0}")
print(f"Total combinations checked: {len(mismatch_df)}")

Maximum denominator difference: 0.0
All denominators consistent: True
Total combinations checked: 240
